# SREDT Venter Bootstrap Re-Analysis with Intercept Reclassification

This notebook demonstrates the **SREDT Venter Re-Analysis** evaluation, which rigorously re-examines
the Semantic Risk Event Development Triangle (SREDT) experiment from iter_1 by applying three
contributions beyond the original analysis:

1. **Venter Intercept Reclassification** — Corrects the prior CV-only criterion by applying the
   full Venter (1998) combined criterion: a transition is `chain_ladder_valid` only if BOTH
   `CV < 0.30` AND the OLS intercept is non-significant (`|a| < 2·SE(a)`).

2. **Bootstrap Confidence Intervals** (B=1000, seed=42) — 95% CIs on development factors `f_j`,
   coefficient of variation CV, and OLS intercepts for each level transition.

3. **Sector-Stratified OLS** — Separate Venter analysis for energy and financials sectors.

**Overall Verdict: DISCONFIRMED** — 0 of 4 transitions meet the combined Venter criterion.


## Setup: Install Dependencies

Installs `loguru` (not pre-installed on Colab). Core scientific packages (numpy, pandas, scipy,
matplotlib) are pre-installed on Colab and only installed locally to match Colab's exact versions.

In [ ]:
import subprocess, sys
def _pip(*a): subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *a])

# loguru — NOT pre-installed on Colab, always install
_pip('loguru==0.7.2')

# Core packages — pre-installed on Colab; install locally to match Colab env
if 'google.colab' not in sys.modules:
    _pip('numpy==2.0.2', 'pandas==2.2.2', 'scipy==1.16.3', 'matplotlib==3.10.0')

print('Dependencies ready.')

## Imports

Imports from the original `eval.py` script, plus `IPython.display` for rendering figures inline.

In [ ]:
import sys
import os
import json
import gc
import math
from pathlib import Path

from loguru import logger
import numpy as np
import pandas as pd
from scipy import stats
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from IPython.display import Image, display

logger.remove()
logger.add(sys.stdout, level='INFO', format='{time:HH:mm:ss}|{level:<7}|{message}')
Path('logs').mkdir(exist_ok=True)
logger.add('logs/eval.log', rotation='30 MB', level='DEBUG')

## Data Loading

Loads `mini_demo_data.json` from GitHub (or falls back to local file). The data contains
40 training scenarios and 20 test scenarios from the SREDT GDELT experiment (iter_1),
plus the original Venter diagnostics and metrics.

In [ ]:
GITHUB_DATA_URL = "https://raw.githubusercontent.com/AMGrobelnik/ai-invention-8ea2f1-semantic-risk-evidence-development-trian/main/round-2/evaluation-1/demo/mini_demo_data.json"

def load_data():
    try:
        import urllib.request
        with urllib.request.urlopen(GITHUB_DATA_URL) as response:
            return json.loads(response.read().decode())
    except Exception: pass
    if os.path.exists('mini_demo_data.json'):
        with open('mini_demo_data.json') as f: return json.load(f)
    raise FileNotFoundError('Could not load mini_demo_data.json')

In [ ]:
data = load_data()

train_rows = data['train_scenario_analysis']
test_rows = data['test_scenario_scores']
venter_raw = data['venter_diagnostics']['level_transitions']
metrics_raw = data['metrics']

train_df = pd.DataFrame(train_rows)
test_df = pd.DataFrame(test_rows)
logger.info(f'Train: {len(train_df)} rows, Test: {len(test_df)} rows')
train_df.head(3)

## Config

All tunable parameters in one place. Set to **minimum values** for a fast demo run.
Original production values are shown in comments.

In [ ]:
# Bootstrap iterations — original: B=1000; minimum demo: B=10
B = 100  # set to 1000 for full reproduction

# Random seed for reproducibility
BOOTSTRAP_SEED = 42

# Output directory for figures
FIGURES_DIR = Path('figures')
FIGURES_DIR.mkdir(exist_ok=True)

print(f'Bootstrap B={B}, seed={BOOTSTRAP_SEED}')
print(f'Figures will be saved to: {FIGURES_DIR.resolve()}')

## Step 1: Data Quality Checks

Checks for degenerate features that would affect the analysis. Key findings:
- `L0` is constant (all = 3.044 = log(21)) because GDELT retrieval fell back to a synthetic
  20-article set for every scenario.
- `keyword_freq` = 1.0 for all scenarios (degenerate baseline).
- 0 valid LLM judge labels (all rate-limited → label = -1).

These flags mark the data as `SYNTHETIC_DATA`, which limits what SC2–SC5 can measure.

In [ ]:
logger.info('Computing data quality checks...')
dq: dict = {}
l0_vals = train_df['l0'].values.astype(float)
dq['l0_std'] = float(np.std(l0_vals))
dq['l0_constant'] = bool(np.std(l0_vals) < 0.001)
dq['l0_unique_values'] = sorted(set(map(float, np.unique(l0_vals))))
dq['n_articles_unique'] = sorted(set(map(int, train_df['n_articles'].unique())))
dq['n_articles_constant'] = bool(train_df['n_articles'].nunique() == 1)
dq['keyword_freq_unique'] = sorted(set(map(float, train_df['keyword_freq'].unique())))
dq['keyword_freq_constant'] = bool(train_df['keyword_freq'].nunique() == 1)
dq['unique_company_risk_pairs'] = int(train_df.groupby(['company', 'risk_type']).ngroups)
dq['n_train'] = len(train_df)
dq['repetition_ratio'] = round(dq['n_train'] / max(1, dq['unique_company_risk_pairs']), 2)
dq['n_valid_llm_labels'] = int((test_df['llm_judge_label'] != -1).sum())
dq['all_labels_missing'] = bool(dq['n_valid_llm_labels'] == 0)
dq['keyword_freq_std_train'] = float(np.std(train_df['keyword_freq'].values.astype(float)))
if dq['l0_constant'] and dq['keyword_freq_constant']:
    dq['flag'] = 'SYNTHETIC_DATA'
elif dq['n_valid_llm_labels'] == 0:
    dq['flag'] = 'NO_GROUND_TRUTH'
else:
    dq['flag'] = 'OK'
logger.info(f"Data quality flag: {dq['flag']}, n_valid_labels={dq['n_valid_llm_labels']}")
print('Data quality:', json.dumps(dq, indent=2))

## Step 2: OLS per Level Transition (Full Training Set)

For each of the 4 development transitions (L0→L1, L1→L2, L2→L3, L3→L4), computes:
- The chain-ladder development factor `f_j = ΣC(i,j+1) / ΣC(i,j)`
- The coefficient of variation (CV) of individual development factors
- OLS regression: `C(i,j+1) = a + b·C(i,j)` — intercept `a` and its standard error
- Venter (1998) verdict under the **corrected combined criterion** (CV < 0.30 AND |a| < 2·SE(a))

The key correction over iter_1: L3→L4 was labeled `chain_ladder_valid` (CV=0.2939 < 0.30) but
its intercept t-statistic = 12.05 (p≈0) makes it a mandatory `factor_plus_constant`.

In [ ]:
def compute_ols_diagnostics(x_vals: np.ndarray, y_vals: np.ndarray, label: str) -> dict:
    """Run OLS on one level transition and return full Venter diagnostics."""
    x = np.array(x_vals, dtype=float)
    y = np.array(y_vals, dtype=float)
    n = len(x)

    degenerate = np.std(x) < 1e-10

    if degenerate:
        f_j = float(np.sum(y) / np.sum(x)) if np.sum(x) > 0 else float('nan')
        return {
            'transition': label, 'n': n,
            'f_j': f_j, 'cv': None, 'individual_ratios': [],
            'slope': None, 'intercept': None, 'se_intercept': None,
            't_stat': None, 'p_value': None, 'intercept_significant': None,
            'r_squared': None, 'verdict_cv_only': 'degenerate', 'verdict_corrected': 'degenerate',
        }

    # Development factors
    individual_ratios = y / x
    f_j = float(np.sum(y) / np.sum(x))
    mean_r = float(np.mean(individual_ratios))
    cv = float(np.std(individual_ratios) / mean_r) if mean_r > 0 else float('nan')

    # OLS: C(i,j+1) = a + b * C(i,j)
    slope, intercept, r_value, _, _ = stats.linregress(x, y)
    r_squared = float(r_value ** 2)

    # SE of intercept
    y_pred = slope * x + intercept
    residuals = y - y_pred
    mse = float(np.sum(residuals ** 2) / (n - 2)) if n > 2 else float('nan')
    x_mean = float(np.mean(x))
    ss_x = float(np.sum((x - x_mean) ** 2))
    if ss_x > 0 and not math.isnan(mse):
        se_intercept = float(math.sqrt(mse * (1.0 / n + x_mean ** 2 / ss_x)))
    else:
        se_intercept = float('nan')

    if not math.isnan(se_intercept) and se_intercept > 0:
        t_stat = float(intercept / se_intercept)
        p_val = float(2 * (1 - stats.t.cdf(abs(t_stat), df=n - 2)))
        intercept_significant = bool(abs(intercept) >= 2 * se_intercept)
    else:
        t_stat = None
        p_val = None
        intercept_significant = None

    # Verdicts
    if cv is None or math.isnan(cv):
        verdict_cv_only = 'degenerate'
        verdict_corrected = 'degenerate'
    elif intercept_significant:
        verdict_cv_only = 'chain_ladder_valid' if cv < 0.30 else ('borderline' if cv <= 0.50 else 'bf_fallback')
        verdict_corrected = 'factor_plus_constant'
    elif cv < 0.30:
        verdict_cv_only = 'chain_ladder_valid'
        verdict_corrected = 'chain_ladder_valid'
    elif cv <= 0.50:
        verdict_cv_only = 'borderline'
        verdict_corrected = 'borderline_bf_preferred'
    else:
        verdict_cv_only = 'bf_fallback'
        verdict_corrected = 'bf_fallback'

    return {
        'transition': label, 'n': n,
        'f_j': f_j, 'cv': cv,
        'individual_ratios': individual_ratios.tolist(),
        'slope': float(slope), 'intercept': float(intercept),
        'se_intercept': se_intercept if not math.isnan(se_intercept) else None,
        't_stat': t_stat, 'p_value': p_val,
        'intercept_significant': intercept_significant,
        'r_squared': r_squared,
        'verdict_cv_only': verdict_cv_only,
        'verdict_corrected': verdict_corrected,
    }

In [ ]:
logger.info('Running OLS for all transitions (full training set)...')
transitions_full = []
for j_label, col_j, col_j1 in [
    ('L0→L1', 'l0', 'l1'), ('L1→L2', 'l1', 'l2'),
    ('L2→L3', 'l2', 'l3'), ('L3→L4', 'l3', 'l4'),
]:
    diag = compute_ols_diagnostics(
        train_df[col_j].values, train_df[col_j1].values, j_label
    )
    transitions_full.append(diag)
    logger.info(
        f"  {j_label}: f_j={diag['f_j']:.4f}, CV={diag['cv']}, "
        f"intercept={diag['intercept']}, t={diag['t_stat']}, "
        f"verdict={diag['verdict_corrected']}"
    )

## Step 3: Sector-Stratified Venter Analysis

Repeats the OLS analysis separately for the **energy** (scen_000–019, n=20) and
**financials** (scen_030–049, n=20) sectors. Each sector has ~6 unique company-risk-type
templates. Both sectors show `factor_plus_constant` verdicts across all transitions.

In [ ]:
def run_sector_analysis(train_df: pd.DataFrame) -> dict:
    """Sector-stratified Venter OLS for energy and financials."""
    sector_diagnostics = {}
    for sector in ['energy', 'financials']:
        sdf = train_df[train_df['sector'] == sector].copy()
        n_sector = len(sdf)
        logger.info(f'Sector {sector}: n={n_sector}')

        if n_sector < 8:
            sector_diagnostics[sector] = {'error': f'Insufficient n={n_sector}', 'n': n_sector}
            continue

        sector_diags = []
        for j_label, col_j, col_j1 in [
            ('L1→L2', 'l1', 'l2'), ('L2→L3', 'l2', 'l3'), ('L3→L4', 'l3', 'l4'),
        ]:
            x = sdf[col_j].values
            y = sdf[col_j1].values
            diag = compute_ols_diagnostics(x, y, j_label)
            diag['n_unique_x'] = int(len(np.unique(np.round(x, 6))))
            sector_diags.append(diag)

        sector_diagnostics[sector] = {
            'n_total': n_sector,
            'n_unique_company_risk': int(sdf.groupby(['company', 'risk_type']).ngroups),
            'transitions': sector_diags,
        }
    return sector_diagnostics


logger.info('Running sector-stratified Venter analysis...')
sector_diagnostics = run_sector_analysis(train_df)
for sector, sd in sector_diagnostics.items():
    verdicts = [t['verdict_corrected'] for t in sd.get('transitions', [])]
    print(f'{sector}: n={sd["n_total"]}, verdicts={verdicts}')

## Step 4: Bootstrap Confidence Intervals

Computes 95% bootstrap CIs (B iterations, seed=42) for `f_j`, CV, and OLS intercept on each
transition. Key result: the L3→L4 intercept CI excludes zero entirely, confirming significance.
The jackknife LOO on L3→L4 shows intercept std = 0.015 (not a single outlier).

> **Config**: `B` is set in the config cell above. Original: B=1000. Minimum demo: B=10.

In [ ]:
def run_bootstrap(train_df: pd.DataFrame, transitions_full: list, B: int = 1000) -> list:
    """Bootstrap CIs (B=1000, seed=42) for f_j, CV, intercept."""
    rng = np.random.default_rng(BOOTSTRAP_SEED)
    bootstrap_ci = []

    for j_label, col_j, col_j1 in [
        ('L0→L1', 'l0', 'l1'), ('L1→L2', 'l1', 'l2'),
        ('L2→L3', 'l2', 'l3'), ('L3→L4', 'l3', 'l4'),
    ]:
        x_all = train_df[col_j].values.astype(float)
        y_all = train_df[col_j1].values.astype(float)
        n = len(x_all)

        boot_fj, boot_cv, boot_intercept = [], [], []
        for _ in range(B):
            idx = rng.integers(0, n, size=n)
            xb, yb = x_all[idx], y_all[idx]

            fj_b = float(np.sum(yb) / np.sum(xb)) if np.sum(xb) > 0 else float('nan')
            boot_fj.append(fj_b)

            if np.std(xb) > 1e-10 and np.all(xb > 0):
                ratios_b = yb / xb
                cv_b = float(np.std(ratios_b) / np.mean(ratios_b)) if np.mean(ratios_b) > 0 else float('nan')
            else:
                cv_b = float('nan')
            boot_cv.append(cv_b)

            if np.std(xb) > 1e-10:
                _, int_b, _, _, _ = stats.linregress(xb, yb)
                boot_intercept.append(float(int_b))
            else:
                boot_intercept.append(float('nan'))

        boot_fj_arr = np.array(boot_fj)
        boot_cv_arr = np.array(boot_cv)
        boot_int_arr = np.array(boot_intercept)

        obs_diag = next(d for d in transitions_full if d['transition'] == j_label)
        obs_intercept = obs_diag['intercept']

        # Bootstrap p-value (two-sided): how often |boot_int - mean(boot_int)| >= |obs - mean(boot_int)|
        if obs_intercept is not None:
            valid_bi = boot_int_arr[~np.isnan(boot_int_arr)]
            if len(valid_bi) > 0:
                mu = float(np.mean(valid_bi))
                p_boot = float(np.mean(np.abs(valid_bi - mu) >= abs(obs_intercept - mu)))
            else:
                p_boot = None
        else:
            p_boot = None

        def pct(arr, q):
            v = arr[~np.isnan(arr)]
            return float(np.percentile(v, q)) if len(v) > 0 else None

        incl_zero = None
        v = boot_int_arr[~np.isnan(boot_int_arr)]
        if len(v) > 0:
            lo, hi = float(np.percentile(v, 2.5)), float(np.percentile(v, 97.5))
            incl_zero = bool(lo <= 0 <= hi)

        bootstrap_ci.append({
            'transition': j_label,
            'f_j_observed': obs_diag['f_j'],
            'f_j_lower': pct(boot_fj_arr, 2.5),
            'f_j_upper': pct(boot_fj_arr, 97.5),
            'cv_observed': obs_diag['cv'],
            'cv_lower': pct(boot_cv_arr, 2.5),
            'cv_upper': pct(boot_cv_arr, 97.5),
            'intercept_observed': obs_intercept,
            'intercept_lower': pct(boot_int_arr, 2.5),
            'intercept_upper': pct(boot_int_arr, 97.5),
            'intercept_p_bootstrap': p_boot,
            'intercept_includes_zero_in_95ci': incl_zero,
        })
        logger.info(f"Bootstrap {j_label}: f_j=[{pct(boot_fj_arr,2.5):.4f}, {pct(boot_fj_arr,97.5):.4f}]")

    return bootstrap_ci


logger.info(f'Running bootstrap (B={B})...')
bootstrap_ci = run_bootstrap(train_df, transitions_full, B=B)

## Step 5: Robustness Checks

Two robustness checks for the L3→L4 transition (the key reclassification):
- **Jackknife LOO**: leave-one-out stability of `f_j` and intercept. Low intercept std confirms
  the high intercept is not driven by a single outlier.
- **Sector swap test**: fit OLS on energy, predict on financials, and vice versa. Negative
  transfer R² shows the sectors have distinct development dynamics.

In [ ]:
def jackknife_l3_l4(train_df: pd.DataFrame) -> dict:
    """Jackknife leave-one-out for L3→L4 f_j and intercept."""
    x_all = train_df['l3'].values.astype(float)
    y_all = train_df['l4'].values.astype(float)
    n = len(x_all)

    jk_fj, jk_int = [], []
    for i in range(n):
        mask = np.ones(n, dtype=bool)
        mask[i] = False
        xi, yi = x_all[mask], y_all[mask]
        fj = float(np.sum(yi) / np.sum(xi)) if np.sum(xi) > 0 else float('nan')
        jk_fj.append(fj)
        if np.std(xi) > 1e-10:
            _, int_b, _, _, _ = stats.linregress(xi, yi)
            jk_int.append(float(int_b))
        else:
            jk_int.append(float('nan'))

    return {
        'n': n,
        'fj_mean': float(np.nanmean(jk_fj)),
        'fj_std': float(np.nanstd(jk_fj)),
        'fj_min': float(np.nanmin(jk_fj)),
        'fj_max': float(np.nanmax(jk_fj)),
        'intercept_mean': float(np.nanmean(jk_int)),
        'intercept_std': float(np.nanstd(jk_int)),
        'intercept_min': float(np.nanmin(jk_int)),
        'intercept_max': float(np.nanmax(jk_int)),
    }


def sector_swap_test(train_df: pd.DataFrame) -> dict:
    """Fit L3→L4 OLS on energy, predict on financials, and vice versa."""
    results = {}
    for fit_sector, pred_sector in [('energy', 'financials'), ('financials', 'energy')]:
        fit_df = train_df[train_df['sector'] == fit_sector]
        pred_df = train_df[train_df['sector'] == pred_sector]
        x_fit = fit_df['l3'].values.astype(float)
        y_fit = fit_df['l4'].values.astype(float)
        x_pred = pred_df['l3'].values.astype(float)
        y_pred_true = pred_df['l4'].values.astype(float)

        if np.std(x_fit) < 1e-10:
            results[f'fit_{fit_sector}_pred_{pred_sector}'] = {'error': 'degenerate x in fit sector'}
            continue

        slope, intercept, r_val, _, _ = stats.linregress(x_fit, y_fit)
        y_pred_hat = slope * x_pred + intercept
        ss_res = float(np.sum((y_pred_true - y_pred_hat) ** 2))
        ss_tot = float(np.sum((y_pred_true - np.mean(y_pred_true)) ** 2))
        r2_transfer = float(1 - ss_res / ss_tot) if ss_tot > 0 else float('nan')

        results[f'fit_{fit_sector}_pred_{pred_sector}'] = {
            'fit_slope': float(slope), 'fit_intercept': float(intercept),
            'transfer_r_squared': r2_transfer,
            'n_fit': int(len(x_fit)), 'n_pred': int(len(x_pred)),
        }
    return results


logger.info('Running robustness checks...')
robustness = {}
try:
    robustness['jackknife_l3_l4'] = jackknife_l3_l4(train_df)
    logger.info(f"Jackknife L3→L4: {robustness['jackknife_l3_l4']}")
except Exception as e:
    logger.error(f'Jackknife failed: {e}')
    robustness['jackknife_l3_l4'] = {'error': str(e)}

try:
    robustness['sector_swap_test'] = sector_swap_test(train_df)
    logger.info(f"Sector swap: {robustness['sector_swap_test']}")
except Exception as e:
    logger.error(f'Sector swap failed: {e}')
    robustness['sector_swap_test'] = {'error': str(e)}

## Step 6: Intercept Significance Table

Compares the corrected Venter verdict against the prior experiment's verdict to identify
reclassifications. The key reclassification is L3→L4: `chain_ladder_valid` in iter_1
→ `factor_plus_constant` after applying the combined Venter criterion.

In [ ]:
prior_verdict_map = {v['j_label']: v['verdict'] for v in venter_raw}

intercept_significance_table = []
for diag in transitions_full:
    prior_v = prior_verdict_map.get(diag['transition'], 'not_in_prior')
    reclassified = (prior_v != diag['verdict_corrected']) and (prior_v != 'not_in_prior')
    intercept_significance_table.append({
        'transition': diag['transition'],
        'intercept': diag['intercept'],
        'se_intercept': diag['se_intercept'],
        't_stat': diag['t_stat'],
        'p_value': diag['p_value'],
        'intercept_significant_2se': diag['intercept_significant'],
        'cv': diag['cv'],
        'verdict_prior_experiment': prior_v,
        'verdict_cv_only': diag['verdict_cv_only'],
        'verdict_corrected': diag['verdict_corrected'],
        'reclassification': reclassified,
    })
    if reclassified:
        logger.warning(
            f"RECLASSIFICATION: {diag['transition']} "
            f"prior={prior_v} → corrected={diag['verdict_corrected']}"
        )

pd.DataFrame(intercept_significance_table)[[
    'transition', 'cv', 't_stat', 'intercept_significant_2se',
    'verdict_prior_experiment', 'verdict_corrected', 'reclassification'
]]

## Step 7: Hypothesis Verdict Table

Evaluates each success criterion (SC1–SC5):
- **SC1** (Venter proportionality): DISCONFIRMED — 0/4 transitions meet combined criterion
- **SC2** (Spearman improvement): INSUFFICIENT_DATA — 0 valid LLM judge labels
- **SC3** (Brier improvement): INSUFFICIENT_DATA — 0 valid labels
- **SC4** (Cohen kappa): NOT_TESTED — human annotation not performed
- **SC5** (L0-L4 rank corr): INSUFFICIENT_DATA — L0 constant, Spearman undefined

In [ ]:
# L0-L4 rank correlation
l0_train = train_df['l0'].values.astype(float)
l4_train = train_df['l4'].values.astype(float)
if np.std(l0_train) < 1e-10:
    l0_l4_spearman = None
    l0_l4_note = 'L0 constant (all 20 articles, synthetic fallback) — Spearman undefined'
else:
    r_sp, _ = stats.spearmanr(l0_train, l4_train)
    l0_l4_spearman = float(r_sp)
    l0_l4_note = 'Spearman rank correlation between L0 and L4 (training set)'

sc1_passes_corrected = [d for d in transitions_full if d['verdict_corrected'] == 'chain_ladder_valid']
sc1_passes_cv_only = [d for d in transitions_full
                      if d['cv'] is not None and not math.isnan(d['cv']) and d['cv'] < 0.30]
reclassifications = [(d['transition'], d['verdict_corrected'])
                     for d in intercept_significance_table if d['reclassification']]

n_valid = dq['n_valid_llm_labels']

hypothesis_verdict = [
    {
        'criterion': 'SC1_venter_proportionality',
        'description': 'CV<0.30 for >=1 transition in >=1 sector, AND intercept non-significant (Venter 1998 combined criterion)',
        'result': {
            'transitions_cv_below_030': [d['transition'] for d in sc1_passes_cv_only],
            'chain_ladder_valid_corrected': [d['transition'] for d in sc1_passes_corrected],
            'reclassifications': reclassifications,
        },
        'threshold': 'CV<0.30 AND |a|<2*SE(a)',
        'verdict': 'DISCONFIRMED' if len(sc1_passes_corrected) == 0 else 'CONFIRMED',
        'note': (
            f'L3→L4 incorrectly labeled chain_ladder_valid in prior experiment '
            f'(CV=0.2939<0.30) but intercept=1.573 with p≈0 requires reclassification '
            f'as factor_plus_constant per Venter (1998). After reclassification, '
            f'{len(sc1_passes_corrected)} of 4 transitions qualify as chain_ladder_valid.'
        ),
    },
    {
        'criterion': 'SC2_spearman_improvement',
        'description': 'SREDT Spearman correlation with ground truth >= flat baseline + 0.15',
        'result': {'n_valid_labels': n_valid, 'ground_truth_source': metrics_raw.get('ground_truth_source')},
        'threshold': 'Spearman improvement >= 0.15',
        'verdict': 'INSUFFICIENT_DATA',
        'note': (
            f'{n_valid} valid LLM judge labels (all rate_limited). '
            'Prior metrics used inadmissible median_split_proxy ground truth.'
        ),
    },
    {
        'criterion': 'SC3_brier_improvement',
        'description': 'SREDT Brier score lower than flat embedding baseline',
        'result': {'n_valid_labels': n_valid},
        'threshold': 'Brier_sredt < Brier_flat',
        'verdict': 'INSUFFICIENT_DATA',
        'note': 'No valid ground truth labels. Prior Brier metrics against synthetic proxy labels are invalid.',
    },
    {
        'criterion': 'SC4_cohen_kappa',
        'description': 'Inter-annotator Cohen kappa >= 0.60 for 3 domain annotators',
        'result': {'annotation_status': 'not_performed'},
        'threshold': 'kappa >= 0.60',
        'verdict': 'NOT_TESTED',
        'note': 'Human annotation deferred to future work.',
    },
    {
        'criterion': 'SC5_circularity_check',
        'description': 'L0-L4 Spearman rank correlation < 0.80 (SREDT not reducible to flat retrieval)',
        'result': {'l0_l4_rank_corr': l0_l4_spearman, 'reason': l0_l4_note},
        'threshold': 'rank_corr < 0.80',
        'verdict': 'INSUFFICIENT_DATA',
        'note': 'L0 is identical for all 40 training scenarios (3.044 = log(21), synthetic). Cannot compute Spearman rank correlation.',
    },
]

for hv in hypothesis_verdict:
    print(f"{hv['criterion']:35s}  {hv['verdict']}")

## Step 8: Publication Figures

Generates 4 figures:
1. **Development triangle heatmap** (60×5, train+test, viridis)
2. **Venter regression scatter plots** (2×2 grid, sector-colored, 95% CI ribbons)
3. **Test scenario score comparison** (SREDT vs flat vs keyword, horizontal bars)
4. **Article count distribution** (reveals synthetic fallback spike at n=20)

In [ ]:
def make_fig1_heatmap(train_df: pd.DataFrame, test_df: pd.DataFrame, out_dir: Path) -> str:
    """Development triangle heatmap."""
    fig, ax = plt.subplots(figsize=(12, 16))
    levels = ['L0', 'L1', 'L2', 'L3', 'L4']
    n_train = len(train_df)
    n_test = len(test_df)

    train_matrix = train_df[['l0', 'l1', 'l2', 'l3', 'l4']].values.astype(float)

    test_matrix = np.full((n_test, 5), np.nan)
    test_matrix[:, 0] = np.log1p(test_df['n_articles'].values)
    # L3/L4 projected (latent but available)
    if 'projected_l3' in test_df.columns:
        test_matrix[:, 3] = test_df['projected_l3'].values
    if 'projected_l4' in test_df.columns:
        test_matrix[:, 4] = test_df['projected_l4'].values

    full_matrix = np.vstack([train_matrix, test_matrix])
    vmax = float(np.nanmax(full_matrix))

    im = ax.imshow(full_matrix, aspect='auto', cmap='viridis', vmin=0, vmax=vmax)

    # Grey out test L1/L2 (not available) and mark L3/L4 as projected
    for row in range(n_train, n_train + n_test):
        for col in [1, 2]:  # L1/L2 not available for test
            ax.add_patch(mpatches.Rectangle(
                (col - 0.5, row - 0.5), 1, 1,
                fill=True, color='lightgrey', alpha=0.85, zorder=2
            ))
        for col in [3, 4]:  # L3/L4 are projected
            ax.add_patch(mpatches.Rectangle(
                (col - 0.5, row - 0.5), 1, 1,
                fill=False, edgecolor='red', linewidth=1.2, zorder=3, linestyle='--'
            ))

    # Train/test separator
    ax.axhline(n_train - 0.5, color='red', linewidth=2.5)
    ax.text(4.6, n_train - 0.5, 'Train/Test split', color='red', fontsize=7, va='center')

    plt.colorbar(im, ax=ax, label='Relevance-Density Mass C(i,j)')

    all_companies = list(train_df['company']) + list(test_df['company'])
    all_sectors = list(train_df['sector']) + list(test_df['sector'])
    sector_colors = ['#1f77b4' if s == 'energy' else '#2ca02c' for s in all_sectors]

    ax.set_yticks(range(n_train + n_test))
    ax.set_yticklabels([f'{c}' for c in all_companies], fontsize=5)
    for tick, color in zip(ax.get_yticklabels(), sector_colors):
        tick.set_color(color)

    ax.set_xticks(range(5))
    ax.set_xticklabels(levels, fontsize=11)
    ax.set_xlabel('Level', fontsize=11)
    ax.set_ylabel('Scenario (blue=energy, green=financials)', fontsize=9)
    ax.set_title('SREDT Development Triangle\n(grey=missing, dashed-red=projected/latent)', fontsize=12)

    energy_patch = mpatches.Patch(color='#1f77b4', label='Energy')
    fin_patch = mpatches.Patch(color='#2ca02c', label='Financials')
    ax.legend(handles=[energy_patch, fin_patch], loc='upper right', fontsize=8)

    ax.text(0.5, -0.03,
        'NOTE: All L0=3.044 (log(21)) — synthetic article fallback, NOT real GDELT data',
        transform=ax.transAxes, ha='center', fontsize=8, color='red',
        bbox=dict(boxstyle='round,pad=0.2', fc='lightyellow', ec='red', alpha=0.8))

    plt.tight_layout()
    out = out_dir / 'fig1_development_triangle_heatmap.png'
    plt.savefig(out, dpi=150, bbox_inches='tight')
    plt.close()
    logger.info(f'Saved {out}')
    return str(out)


fig1_path = make_fig1_heatmap(train_df, test_df, FIGURES_DIR)
display(Image(fig1_path, width=700))

In [ ]:
def make_fig2_venter_scatter(train_df: pd.DataFrame, transitions_full: list, out_dir: Path) -> str:
    """2x2 Venter regression scatter plots."""
    fig, axes = plt.subplots(2, 2, figsize=(13, 10))
    axes = axes.flatten()

    transition_cols = [
        ('L0→L1', 'l0', 'l1'), ('L1→L2', 'l1', 'l2'),
        ('L2→L3', 'l2', 'l3'), ('L3→L4', 'l3', 'l4'),
    ]
    sector_colors = {'energy': '#1f77b4', 'financials': '#2ca02c'}

    for ax, (j_label, col_j, col_j1) in zip(axes, transition_cols):
        diag = next(d for d in transitions_full if d['transition'] == j_label)
        x = train_df[col_j].values.astype(float)
        y = train_df[col_j1].values.astype(float)
        colors = [sector_colors[s] for s in train_df['sector']]

        ax.scatter(x, y, c=colors, alpha=0.7, s=40, zorder=3)

        # Company initials
        for xi, yi, co in zip(x, y, train_df['company']):
            ax.annotate(co[:2], (xi, yi), fontsize=4, ha='center', va='bottom', alpha=0.6)

        # OLS line
        if diag['slope'] is not None and np.std(x) > 1e-10:
            x_range = np.linspace(x.min(), x.max(), 200)
            y_hat = diag['slope'] * x_range + diag['intercept']
            ax.plot(x_range, y_hat, 'k-', linewidth=1.5, label='OLS fit', zorder=4)

            # 95% CI ribbon
            n = len(x)
            x_mean = np.mean(x)
            ss_x = np.sum((x - x_mean) ** 2)
            y_pred = diag['slope'] * x + diag['intercept']
            mse = np.sum((y - y_pred) ** 2) / (n - 2) if n > 2 else 0
            if ss_x > 0:
                se_line = np.sqrt(mse * (1 / n + (x_range - x_mean) ** 2 / ss_x))
                t_crit = float(stats.t.ppf(0.975, df=n - 2))
                ax.fill_between(x_range, y_hat - t_crit * se_line, y_hat + t_crit * se_line,
                                alpha=0.15, color='grey', label='95% CI')

        cv_str = f"{diag['cv']:.4f}" if diag['cv'] is not None else 'N/A'
        t_str = f"{diag['t_stat']:.3f}" if diag['t_stat'] is not None else 'N/A'
        p_str = f"{diag['p_value']:.3g}" if diag['p_value'] is not None else 'N/A'
        int_str = f"{diag['intercept']:.4f}" if diag['intercept'] is not None else 'N/A'
        title_color = 'red' if j_label == 'L3→L4' else 'black'
        ax.set_title(
            f"{j_label}  |  CV={cv_str}  intercept={int_str}\n"
            f"t={t_str}  p={p_str}  verdict={diag['verdict_corrected']}",
            fontsize=8, color=title_color
        )
        if j_label == 'L3→L4':
            ax.text(0.02, 0.96,
                '⚠ Intercept significant → reclassified\nfactor_plus_constant (was chain_ladder_valid)',
                transform=ax.transAxes, fontsize=7, color='red', va='top',
                bbox=dict(boxstyle='round', fc='lightyellow', ec='red', alpha=0.9))

        col_names = {'l0': 'L0', 'l1': 'L1', 'l2': 'L2', 'l3': 'L3', 'l4': 'L4'}
        ax.set_xlabel(f"C(i,j) = {col_names[col_j]}", fontsize=9)
        ax.set_ylabel(f"C(i,j+1) = {col_names[col_j1]}", fontsize=9)
        ax.legend(fontsize=7)
        ax.grid(True, alpha=0.3)

    energy_patch = mpatches.Patch(color='#1f77b4', label='Energy')
    fin_patch = mpatches.Patch(color='#2ca02c', label='Financials')
    fig.legend(handles=[energy_patch, fin_patch], loc='upper center',
               ncol=2, fontsize=9, bbox_to_anchor=(0.5, 1.01))
    plt.suptitle('Venter (1998) Development Factor Regression — All Transitions', fontsize=12, y=1.03)
    plt.tight_layout()
    out = out_dir / 'fig2_venter_regression_scatterplots.png'
    plt.savefig(out, dpi=150, bbox_inches='tight')
    plt.close()
    logger.info(f'Saved {out}')
    return str(out)


fig2_path = make_fig2_venter_scatter(train_df, transitions_full, FIGURES_DIR)
display(Image(fig2_path, width=700))

In [ ]:
def make_fig3_score_comparison(test_df: pd.DataFrame, out_dir: Path) -> str:
    """Horizontal bar chart of test scenario scores."""
    fig, ax = plt.subplots(figsize=(10, 12))

    tdf = test_df.copy().sort_values('projected_l4', ascending=False)
    y_pos = np.arange(len(tdf))
    bar_h = 0.25

    ax.barh(y_pos + bar_h, tdf['projected_l4'], bar_h, color='#1b4f72', label='SREDT projected_l4', alpha=0.85)
    ax.barh(y_pos, tdf['flat_sim'], bar_h, color='#e67e22', label='Flat cosine similarity', alpha=0.85)
    ax.barh(y_pos - bar_h, tdf['keyword_freq'], bar_h, color='#95a5a6', label='Keyword frequency', alpha=0.85)

    labels = [f"{row.company} ({row.sector[:3]})\n{row.risk_type}" for row in tdf.itertuples()]
    ax.set_yticks(y_pos)
    ax.set_yticklabels(labels, fontsize=7)

    # Sector separator
    n_fin = (tdf['sector'] == 'financials').sum()
    n_en = (tdf['sector'] == 'energy').sum()
    if n_fin > 0 and n_en > 0:
        sep_y = n_fin - 0.5
        ax.axhline(sep_y, color='black', linewidth=1.5, linestyle='--', alpha=0.6)
        ax.text(0.01, sep_y + 0.3, 'Financials (above) ↔ Energy (below)', fontsize=7, alpha=0.7)

    ax.set_xlabel('Score', fontsize=10)
    ax.set_title('Test Scenario Score Comparison: SREDT vs Baselines\n'
                 '(No valid ground truth — SC2/SC3 INSUFFICIENT_DATA: all LLM judge calls rate-limited)', fontsize=10)
    ax.legend(fontsize=8, loc='lower right')
    ax.grid(True, axis='x', alpha=0.3)
    plt.tight_layout()
    out = out_dir / 'fig3_test_score_comparison.png'
    plt.savefig(out, dpi=150, bbox_inches='tight')
    plt.close()
    logger.info(f'Saved {out}')
    return str(out)


fig3_path = make_fig3_score_comparison(test_df, FIGURES_DIR)
display(Image(fig3_path, width=700))

In [ ]:
def make_fig4_article_count(train_df: pd.DataFrame, test_df: pd.DataFrame, out_dir: Path) -> str:
    """Article count distribution — reveals synthetic data fallback."""
    fig, axes = plt.subplots(1, 2, figsize=(10, 5))

    for ax, (df, label) in zip(axes, [(train_df, 'Train (40 scenarios)'), (test_df, 'Test (20 scenarios)')]):
        counts = df['n_articles'].values.astype(int)
        unique_counts = sorted(set(counts))
        ax.bar(unique_counts, [np.sum(counts == c) for c in unique_counts],
               color='#2980b9', alpha=0.85, edgecolor='black')
        ax.set_xlabel('n_articles', fontsize=10)
        ax.set_ylabel('Frequency', fontsize=10)
        ax.set_title(f'{label}', fontsize=10)
        ax.set_xticks(unique_counts)
        ax.text(0.5, 0.85, f"All = {unique_counts[0] if len(unique_counts)==1 else '?'}\n(synthetic fallback)",
                transform=ax.transAxes, ha='center', fontsize=9, color='red',
                bbox=dict(boxstyle='round', fc='lightyellow', ec='red', alpha=0.9))

    fig.suptitle(
        'Article Count Distribution: Revealing Synthetic Data Fallback\n'
        'Real data should show skewed distribution; uniform spike at 20 = GDELT fallback',
        fontsize=10
    )
    plt.tight_layout()
    out = out_dir / 'fig4_article_count_distribution.png'
    plt.savefig(out, dpi=150, bbox_inches='tight')
    plt.close()
    logger.info(f'Saved {out}')
    return str(out)


fig4_path = make_fig4_article_count(train_df, test_df, FIGURES_DIR)
display(Image(fig4_path, width=700))

## Results Summary

Final summary table of all hypothesis verdicts and key diagnostic metrics.

In [ ]:
print('=' * 70)
print('SREDT Venter Bootstrap Re-Analysis — Final Results')
print('=' * 70)

print('\n--- Hypothesis Verdicts ---')
for hv in hypothesis_verdict:
    print(f"  {hv['criterion']:35s}  {hv['verdict']}")

print('\n--- Intercept Reclassification Summary ---')
for row in intercept_significance_table:
    flag = ' [RECLASSIFIED]' if row['reclassification'] else ''
    cv_s = f"{row['cv']:.4f}" if row['cv'] is not None else 'N/A (degenerate)'
    t_s = f"{row['t_stat']:.2f}" if row['t_stat'] is not None else 'N/A'
    print(f"  {row['transition']}: CV={cv_s}, t={t_s}, verdict={row['verdict_corrected']}{flag}")

print('\n--- Bootstrap CIs (L3→L4 key transition) ---')
l3l4_boot = next(b for b in bootstrap_ci if b['transition'] == 'L3→L4')
print(f"  f_j: {l3l4_boot['f_j_observed']:.4f} (95% CI: [{l3l4_boot['f_j_lower']:.4f}, {l3l4_boot['f_j_upper']:.4f}])")
print(f"  intercept: {l3l4_boot['intercept_observed']:.4f} (95% CI: [{l3l4_boot['intercept_lower']:.4f}, {l3l4_boot['intercept_upper']:.4f}])")
print(f"  intercept CI includes zero: {l3l4_boot['intercept_includes_zero_in_95ci']}")

print('\n--- Jackknife L3→L4 ---')
jk = robustness['jackknife_l3_l4']
print(f"  intercept mean={jk['intercept_mean']:.4f}, std={jk['intercept_std']:.4f}")

print('\n--- Data Quality ---')
print(f"  Flag: {dq['flag']}")
print(f"  L0 constant: {dq['l0_constant']} (all = {dq['l0_unique_values'][0]:.3f} = log(21))")
print(f"  Valid LLM judge labels: {dq['n_valid_llm_labels']} / {len(test_df)}")

print('\n--- Overall Verdict ---')
print('  DISCONFIRMED')
print('  SC1 (Venter proportionality): 0/4 transitions meet combined CV+intercept criterion.')
print('  SC2-SC5: INSUFFICIENT_DATA or NOT_TESTED (no valid ground truth labels).')

# Bootstrap CI summary table
print('\n--- Bootstrap CIs (all transitions) ---')
rows = []
for b in bootstrap_ci:
    rows.append({
        'transition': b['transition'],
        'f_j_obs': f"{b['f_j_observed']:.4f}",
        'f_j_95CI': f"[{b['f_j_lower']:.4f}, {b['f_j_upper']:.4f}]" if b['f_j_lower'] else 'N/A',
        'int_obs': f"{b['intercept_observed']:.4f}" if b['intercept_observed'] is not None else 'N/A',
        'int_95CI_excl0': str(not b['intercept_includes_zero_in_95ci']) if b['intercept_includes_zero_in_95ci'] is not None else 'N/A',
    })
print(pd.DataFrame(rows).to_string(index=False))